# Control rápido por eventos: estado del arte

**Nivel:** 3  
**Requisitos previos:** Control rápido por eventos nivel 1  
**Tiempo de lectura:** 25 minutos

## Objetivos

El objetivo es explicar el problema y sus datos, inspeccionar y, en niveles superiores, ejecutar el método publicado en su tarea emparejada. El controlador recibe un evento, reduce solo el estado observado, filtra candidatos registrados mediante su contrato de capacidades y emite una decisión inspeccionable. Rápido es un presupuesto operativo, no una etiqueta cerebral ni una afirmación de corrección.

## Preparación

Las versiones siguientes están fijadas en `notebooks/requirements.txt` y también constan en los metadatos de este notebook.

```bash
pip install reflexmesh==0.1.2 numpy==2.4.6 matplotlib==3.11.2 torch==2.14.1 joblib==1.6.0 gymnasium==1.4.0 scikit-learn==1.9.1 stable-baselines3==2.9.2 sb3-contrib==2.9.2
```

## Datos

El notebook lee artefactos comprometidos del producto. Propietario: Neuraxis. Repositorio: https://github.com/fsantibanezleal/CAOS_Neuraxis. Licencia: Apache-2.0. El JSON es un artefacto experimental propio y congelado; este notebook no redistribuye datos de benchmarks externos.

Este notebook lee `data/derived/index.json` y las trazas listadas. La ejecución usa un espacio aislado de caso Reflexmesh. Para métodos aprendidos, descarga los activos públicos exactos con `python -m reflexmesh.cli fetch-models --destination models/checkpoints --version v0.01.000` y define `NEURAXIS_CHECKPOINTS` hacia esa carpeta. El manifiesto de publicación verifica los hashes. M09/M10 se leen desde trazas congeladas, salvo que `NEURAXIS_NOTEBOOK_MODEL` indique explícitamente un modelo Ollama local configurado.

Artefactos de esta lección:

- `data/derived/index.json`
- `data/derived/episodes/M01-C01-nominal-40000.json`
- `data/derived/episodes/M02-C01-nominal-40000.json`

## Método

La FSM protegida del paquete es la referencia ejecutable de esta primera corrida. La traza canónica C01 aporta el evento exacto anterior a la decisión y sus candidatos; el notebook reconstruye los contratos públicos de Reflexmesh y vuelve a pedir una decisión a la política publicada.

\(a_t=\arg\max_{a\in\mathcal A_t^{adm}} U(a\mid o_t)\), con «stop» cuando no queda una acción respaldada. El conjunto autorizado se calcula independientemente de los puntajes aprendidos.

### Comprueba las versiones fijadas del entorno.

In [1]:
# Setup is performed outside the notebook; report the exact runtime pins.
import importlib.metadata
for distribution in ('reflexmesh', 'numpy', 'matplotlib', 'torch', 'joblib', 'gymnasium', 'scikit-learn', 'stable-baselines3', 'sb3-contrib'):
    print('{}=={}'.format(distribution, importlib.metadata.version(distribution)))

reflexmesh==0.1.2
numpy==2.4.6
matplotlib==3.11.2
torch==2.14.1
joblib==1.6.0
gymnasium==1.4.0
scikit-learn==1.9.1
stable-baselines3==2.9.2
sb3-contrib==2.9.2


### Verifica la auditoría y los bytes de cada traza antes de interpretar resultados.

In [2]:
# Lee la evidencia comprometida y usa la API publicada; nunca define otra política.
import hashlib, json
from pathlib import Path
ROOT = Path('../../../').resolve()
index_path = ROOT / 'data/derived/index.json'
index_bytes = index_path.read_bytes()
index = json.loads(index_bytes)
audit = json.loads((ROOT / 'data/derived/benchmark-adequacy.json').read_text(encoding='utf-8'))
assert hashlib.sha256(index_bytes).hexdigest() == audit['index_sha256']
selected_methods = ['M01', 'M02']
sample_case = 'C01'
seed = 40000
records = {}
for method_id in selected_methods:
    row = next(r for r in index['episodes'] if r['method_id'] == method_id and r['case_id'] == sample_case and r['variant'] == 'nominal' and r['seed'] == seed)
    trace_path = ROOT / 'data/derived' / row['path']
    trace_bytes = trace_path.read_bytes()
    assert hashlib.sha256(trace_bytes).hexdigest() == row['sha256']
    records[method_id] = json.loads(trace_bytes)
print('Index digest:', hashlib.sha256(index_bytes).hexdigest())
print('Evidence records:', len(records), '| case:', sample_case, '| seed:', seed)
print('Methods:', ', '.join(selected_methods))

Index digest: 1024a451fdc1f1aebc4e8d8d66e6bf1aca119f26ae058d3de9b84ff312d8d7a6
Evidence records: 2 | case: C01 | seed: 40000
Methods: M01, M02


### Inspecciona las decisiones y resultados originales

In [3]:
# Resume la primera acción registrada, los candidatos autorizados y el resultado del oráculo independiente.
for method_id, episode in records.items():
    first = episode['decisions'][0]
    allowed = [candidate['action'] for candidate in first['candidates'] if candidate['allowed']]
    print(method_id, '| action=', first['action'], '| reason=', first['reason'], '| allowed=', allowed)
    print('  success=', episode['metrics']['task_success'], '| steps=', episode['metrics']['steps'], '| violations=', episode['metrics']['violations'])

M01 | action= a-move | reason= explicit_state_guard | allowed= ['a-ack_event', 'a-move', 'a-refresh', 'a-inspect', 'a-index', 'a-wait_dependency']
  success= 1 | steps= 1 | violations= 0
M02 | action= a-move | reason= reactive_behavior_tree | allowed= ['a-ack_event', 'a-move', 'a-refresh', 'a-inspect', 'a-index', 'a-wait_dependency']
  success= 1 | steps= 1 | violations= 0


### Ejecuta las políticas publicadas y compara la corrida

In [4]:
# Los resultados de planificación se leen de las trazas congeladas; no hay llamada implícita a un LLM.
import os
from pathlib import Path
from reflexmesh.environments import CaseSpec, SoftwareEnvironment
from reflexmesh.pipeline import make_policy
from reflexmesh.planners import OllamaPlanner
checkpoint_dir = Path(os.environ.get('NEURAXIS_CHECKPOINTS', str(ROOT / 'models/checkpoints')))
planner_model = os.environ.get('NEURAXIS_NOTEBOOK_MODEL')
reruns = {}
for method_id in selected_methods:
    if method_id in ('M09', 'M10') and not planner_model:
        print(method_id, 'not rerun: planner needs explicit NEURAXIS_NOTEBOOK_MODEL; frozen result remains above')
        continue
    planner = OllamaPlanner(planner_model) if planner_model and method_id in ('M09', 'M10', 'M11', 'M12') else None
    policy = make_policy(method_id, checkpoint_dir, planner=planner)
    policy.reset('neuraxis-notebook-' + method_id)
    with SoftwareEnvironment(CaseSpec(sample_case, 'nominal', seed)) as env:
        while not env.terminal:
            _, outcome = env.step(policy.predict(env.observe()))
        success = env.verify()
    recorded = records[method_id]
    reruns[method_id] = {'success': success, 'recorded_success': bool(recorded['metrics']['task_success']), 'recorded_first_action': recorded['decisions'][0]['action']}
    print(method_id, reruns[method_id])

M01 {'success': True, 'recorded_success': True, 'recorded_first_action': 'a-move'}
M02 {'success': True, 'recorded_success': True, 'recorded_first_action': 'a-move'}


### Compara la secuencia completa y no solo la primera acción

In [5]:
# La igualdad de secuencia es diagnóstica; la verificación independiente decide el éxito.
for method_id, episode in records.items():
    actions = [decision['action'] for decision in episode['decisions']]
    print(method_id, ' -> '.join(actions), '| terminal=', episode['status'], '| verified=', episode['metrics']['task_success'])

M01 a-move | terminal= succeeded | verified= 1
M02 a-move | terminal= succeeded | verified= 1


### Recalcula resultados por variante desde todas las trazas del caso

In [6]:
# Lee todas las semillas del caso elegido y conserva cada episodio fallido en el denominador.
from collections import defaultdict
groups = defaultdict(list)
for row in index['episodes']:
    if row['case_id'] not in (sample_case,) or row['method_id'] not in selected_methods:
        continue
    raw = (ROOT / 'data/derived' / row['path']).read_bytes()
    assert hashlib.sha256(raw).hexdigest() == row['sha256']
    episode = json.loads(raw)
    groups[(row['method_id'], row['variant'])].append(int(bool(episode['metrics']['task_success'])))
for (method_id, variant), values in sorted(groups.items()):
    print(method_id, variant, 'successes=', sum(values), 'episodes=', len(values), 'rate=', round(sum(values)/len(values), 3))

M01 boundary successes= 10 episodes= 10 rate= 1.0
M01 delayed_observation successes= 10 episodes= 10 rate= 1.0
M01 duplicate_event successes= 10 episodes= 10 rate= 1.0
M01 nominal successes= 10 episodes= 10 rate= 1.0
M01 stale_conflict successes= 10 episodes= 10 rate= 1.0
M01 unavailable_dependency successes= 10 episodes= 10 rate= 1.0
M02 boundary successes= 10 episodes= 10 rate= 1.0
M02 delayed_observation successes= 10 episodes= 10 rate= 1.0
M02 duplicate_event successes= 10 episodes= 10 rate= 1.0
M02 nominal successes= 10 episodes= 10 rate= 1.0
M02 stale_conflict successes= 10 episodes= 10 rate= 1.0
M02 unavailable_dependency successes= 10 episodes= 10 rate= 1.0


### Visualiza el estrés por variante sin mezclar familias

In [7]:
# Cada punto resume las semillas de una sola variante del caso, no el benchmark completo.
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import display
variants = sorted(set(key[1] for key in groups))
fig, ax = plt.subplots(figsize=(9, 4.5), constrained_layout=True)
for method_id in selected_methods:
    rates = [sum(groups[(method_id, variant)])/len(groups[(method_id, variant)]) if groups[(method_id, variant)] else float('nan') for variant in variants]
    ax.plot(variants, rates, marker='o', label=method_id)
ax.set_ylim(-0.05, 1.05)
ax.set_ylabel('Tasa de éxito verificado')
ax.set_title(sample_case + ' por variante')
ax.tick_params(axis='x', rotation=25)
ax.grid(axis='y', alpha=.25)
ax.legend()
display(fig)
plt.close(fig)

<Figure size 900x450 with 1 Axes>

## Estado del arte

Jev es un producto descrito por su proveedor para decisiones rápidas tipadas y sirve como referencia inicial, no como baseline validado aquí de forma independiente. DPT-Agent ya combina control FSM/código con razonamiento LLM asíncrono. La neurociencia también estudia la arbitraje sensible a fiabilidad entre aprendizaje basado en modelo y libre de modelo. Por ello, Neuraxis no afirma inventar System One, las decisiones tipadas ni la composición rápido/lento.

## Resultados

Esta repetición con el paquete comprueba una decisión protegida exacta sobre la misma observación congelada. No demuestra transferencia general, un mecanismo biológico ni superioridad frente a un sistema desplegado y emparejado.

Este paso de autoría no compromete salidas generadas; `scripts/run_notebooks.py` las ejecuta y guarda desde un kernel nuevo.

## Resumen

Una acción admisible es una capacidad acotada, no texto generado. El paquete publicado reproduce la regla seleccionada en una traza real congelada; es una comprobación mecánica, no una afirmación de inteligencia.

## Referencias

- [JEV](https://typesafe.ai/blog/introducing-system-one-models-and-jev)
- [LEE](https://doi.org/10.1016/j.neuron.2013.11.028)
- [DPT](https://aclanthology.org/2025.acl-long.206/)